# CSIRO Image2Biomass · Assignment 1: EDA and baselines

**CENG313 · Group 42**

We predict five dry biomass weights (in grams) from a top-view pasture image. This notebook checks the quality of the data, explores it with the four EDA questions from the course (trust, distributions, extremes, correlation) and evaluates two baseline models.

**How to run.** Open the notebook in Colab, add your Kaggle API token to the 🔑 **Secrets** panel as `KAGGLE_API_TOKEN` (with notebook access enabled) and run the cells in order. Reusable code lives in `src/data.py` and `src/models.py` of the repository. [Kaggle data description](https://www.kaggle.com/competitions/csiro-biomass/data)

## 0. Setup

In [ ]:
# Clone the project repository so the notebook can import the code in src/.
import os

if not os.path.isdir("src"):
    if not os.path.isdir("GROUP42_CENG313"):
        !git clone https://github.com/yavuzhan-kilic/GROUP42_CENG313.git
    %cd GROUP42_CENG313

In [ ]:
import subprocess
import sys
from pathlib import Path
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data import (biomass_differences, image_features, image_sizes,
                      image_table, load_image, metadata_table)
from src.models import (cross_validate, group_folds, kaggle_r2,
                        mean_predictions, rgb_ridge_predictions, split_images)

# Use CSIRO_DATA_DIR if it is set, otherwise the default Colab location.
data_dir = Path(os.environ.get("CSIRO_DATA_DIR", "/content/csiro_data"))

# Download and extract the competition data only if it is not there yet.
if not (data_dir / "train.csv").is_file():
    zip_path = data_dir.parent / "csiro-biomass.zip"
    if not zip_path.is_file():
        from google.colab import userdata

        os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
        subprocess.run(["kaggle", "competitions", "download", "csiro-biomass",
                        "-p", str(zip_path.parent)], check=True)
    with ZipFile(zip_path) as archive:
        archive.extractall(data_dir)

print("Data directory:", data_dir)
print("Files:", sorted(file.name for file in data_dir.iterdir()))

## 1. The data

`train.csv` has one row per **image and target pair**. We first look at its size, columns and types.

In [ ]:
data = pd.read_csv(data_dir / "train.csv")

print("Rows and columns:", data.shape)
display(data.head(10))
display(pd.DataFrame({"Type": data.dtypes, "Unique values": data.nunique()}))

**Variable types** (course week 3–4):

| Column | Type | Note |
|---|---|---|
| `sample_id`, `image_path` | identifier | not features |
| `State`, `Species` | categorical, nominal | no order between categories |
| `Sampling_Date` | numeric, interval scale | differences are meaningful, ratios are not |
| `Pre_GSHH_NDVI` | numeric, continuous index | vegetation index measured before cutting |
| `Height_Ave_cm` | numeric, ratio scale | true zero |
| `target` (five biomass weights) | numeric, ratio scale | 0 g means no material |

The five targets are `Dry_Green_g`, `Dry_Dead_g`, `Dry_Clover_g`, `GDM_g` (green dry matter = green + clover) and `Dry_Total_g` (GDM + dead).

## 2. Tidy data

The raw table is **not tidy**: five different variables are stored in one `target` column and named by `target_name`. We reshape it so that each row is one observation (an image) and each column is one variable. Image metadata goes into a second table with the same index.

In [ ]:
# One row per image, one column per target. pivot() fails on duplicate pairs.
images = image_table(data)

# One row per image with the non-target columns (date, state, species, NDVI, height).
metadata = metadata_table(data)
metadata["Sampling_Date"] = pd.to_datetime(metadata["Sampling_Date"])

print("Raw rows:", len(data))
print("Unique images:", len(images))
print("Rows per image:", data.groupby("image_path").size().unique())

display(images.head())
display(metadata.head())

## 3. Trust: data quality

Before analysing the values we ask how far we can trust them. We look for the problem types from the lecture: duplicates, missing values, erroneous (impossible) values and inconsistent values.

In [ ]:
print("Exact duplicate rows:", data.duplicated().sum())
print("Duplicate sample ids:", data["sample_id"].duplicated().sum())

# Missing and negative (impossible) measurements per target.
display(pd.DataFrame({
    "Missing": images.isna().sum(),
    "Negative": images.lt(0).sum(),
}))

# Missing metadata values per column.
display(metadata.isna().sum().rename("Missing metadata values").to_frame())

In [ ]:
# The labels should satisfy: green + clover = GDM, and GDM + dead = total.
differences = biomass_differences(images)
display(differences.max().round(4).rename("Largest difference (g)").to_frame())

# Review the images where a sum is off by more than 0.1 g.
flagged = differences[differences.gt(0.1).any(axis=1)]
print("Images with a difference above 0.1 g:", len(flagged))
display(images.loc[flagged.index].join(flagged.round(4)))

In [ ]:
# Open every image: compute mean RGB values and list missing or unreadable files.
rgb, missing_images, unreadable_images = image_features(data_dir, images.index)

print("Images checked:", len(images))
print("Missing image files:", len(missing_images))
print("Unreadable image files:", len(unreadable_images))

# Image dimensions.
display(image_sizes(data_dir, images.index).value_counts().rename("Images").to_frame())

**Findings.**

- The 1,785 raw rows become 357 unique images with exactly five rows each. There are no duplicate rows, no missing or negative targets and no missing or unreadable image files.
- One image has an **inconsistent value** (attribute noise): for `ID2014192906.jpg`, GDM + dead = 10.5 g but the recorded total is 10.1912 g, a gap of 0.3088 g. We keep the original values and report the discrepancy, because we cannot tell which of the numbers is wrong.
- We did not remove or impute any value, so the cleaned table contains the original measurements.

✏️ *Add after running: missing metadata values (if any) and the image dimensions.*

## 4. Distributions

How much do the targets vary? Each plot is paired with summary statistics, because a plot alone can mislead.

In [ ]:
# Summary statistics plus skewness (0 = symmetric, positive = long right tail).
summary = images.describe().T
summary["skew"] = images.skew()
display(summary.round(2))

images.hist(bins=20, figsize=(12, 7), layout=(2, 3))
plt.suptitle("Target distributions (grams)")
plt.tight_layout()
plt.show()

In [ ]:
# Exact zeros per target.
zeros = images.eq(0).sum()
display(pd.DataFrame({"Images with 0 g": zeros,
                      "Share (%)": (100 * zeros / len(images)).round(1)}))

In [ ]:
# A log transform compresses the long right tail. log1p(x) = log(1 + x) is defined at 0.
figure, axes = plt.subplots(1, 2, figsize=(11, 3.5))
images["Dry_Total_g"].hist(bins=20, ax=axes[0])
axes[0].set_title(f"Dry_Total_g (skew {images['Dry_Total_g'].skew():.2f})")
np.log1p(images["Dry_Total_g"]).hist(bins=20, ax=axes[1])
axes[1].set_title(f"log1p(Dry_Total_g) (skew {np.log1p(images['Dry_Total_g']).skew():.2f})")
plt.tight_layout()
plt.show()

**Findings.**

- Clover is exactly 0 g in 135 of 357 images, dead material in 40 and green in 18. We treat these zeros as valid measurements (a paddock without clover), not as missing values.
- All five targets have a long tail toward high values (positive skew).

✏️ *Add after running: the skew values, the typical range of total biomass, and whether the log transform makes the distribution more symmetric (an option for Assignment 2).*

## 5. Extremes

Which images have the lowest and highest total biomass, and do the extreme labels look plausible in the image?

In [ ]:
ranked = images["Dry_Total_g"].sort_values()
middle = len(ranked) // 2
selection = {
    "Lowest": ranked.index[:3],
    "Median": ranked.index[middle - 1:middle + 2],
    "Highest": ranked.index[-3:],
}

display(images.round(2).join(metadata)
        .loc[list(selection["Lowest"]) + list(selection["Highest"])])

figure, axes = plt.subplots(3, 3, figsize=(13, 8))
for row, (label, paths) in zip(axes, selection.items()):
    for axis, path in zip(row, paths):
        axis.imshow(load_image(data_dir, path))
        axis.set_title(f"{label}: {images.loc[path, 'Dry_Total_g']:.1f} g", fontsize=10)
        axis.axis("off")
plt.tight_layout()
plt.show()

✏️ **Findings (write after running).** *Describe what you see: how do low and high biomass images differ (density, colour, height of the sward)? Do the highest values look like real measurements or like errors? State whether you keep them and why.*

## 6. Correlation

First between the five targets, then between the image metadata and the targets.

In [ ]:
correlation = images.corr()

figure, axis = plt.subplots(figsize=(6, 5))
heatmap = axis.imshow(correlation, vmin=-1, vmax=1, cmap="coolwarm")
axis.set_xticks(range(len(correlation)), correlation.columns, rotation=45, ha="right")
axis.set_yticks(range(len(correlation)), correlation.index)
for i in range(len(correlation)):
    for j in range(len(correlation)):
        axis.text(j, i, f"{correlation.iloc[i, j]:.2f}", ha="center", va="center")
figure.colorbar(heatmap, label="Pearson correlation")
axis.set_title("Correlation between targets")
plt.tight_layout()
plt.show()

In [ ]:
# Numeric metadata against total biomass.
numeric_columns = ["Pre_GSHH_NDVI", "Height_Ave_cm"]

figure, axes = plt.subplots(1, 2, figsize=(11, 4))
for axis, column in zip(axes, numeric_columns):
    r = metadata[column].corr(images["Dry_Total_g"])
    axis.scatter(metadata[column], images["Dry_Total_g"], s=12, alpha=0.6)
    axis.set_xlabel(column)
    axis.set_ylabel("Dry_Total_g")
    axis.set_title(f"r = {r:.2f}")
plt.tight_layout()
plt.show()

# Correlation of the numeric metadata and the mean colour values with every target.
display(pd.concat([metadata[numeric_columns], rgb], axis=1)
        .apply(lambda column: images.corrwith(column)).round(2))

In [ ]:
# Categorical metadata: number of images and total biomass per category.
table = images.join(metadata)

for column in ["State", "Species"]:
    display(table.groupby(column)["Dry_Total_g"]
            .agg(["count", "mean", "median", "std"]).round(1)
            .sort_values("count", ascending=False))

figure, axis = plt.subplots(figsize=(6, 4))
table.boxplot(column="Dry_Total_g", by="State", ax=axis)
axis.set_title("Total biomass by state")
axis.set_ylabel("Dry_Total_g")
plt.suptitle("")
plt.tight_layout()
plt.show()

✏️ **Findings (write after running).** *Which targets are almost the same variable (for example green and GDM)? Which metadata column is most related to total biomass? Do states or species differ in biomass? Does mean greenness relate to green biomass?*

## 7. Data ≠ truth: what is missing and who is not represented

A clean table is not automatically representative. We check what the test set contains and how the training images are spread over time and place.

In [ ]:
# Which columns does the test set have? Columns missing here cannot be model inputs.
test = pd.read_csv(data_dir / "test.csv")
print("Test rows:", len(test), "| test images:", test["image_path"].nunique())
print("Test columns:", list(test.columns))
print("Train-only columns:", [column for column in data.columns if column not in test.columns])

In [ ]:
# Sampling dates: how many days, and how many images per day and per month?
per_day = metadata["Sampling_Date"].value_counts().sort_index()
print("Sampling days:", len(per_day))
print("From", per_day.index.min().date(), "to", per_day.index.max().date())
print("Images per day: min", per_day.min(), "| median", per_day.median(), "| max", per_day.max())

per_month = metadata["Sampling_Date"].dt.to_period("M").value_counts().sort_index()
per_month.plot.bar(figsize=(11, 3), title="Images per sampling month")
plt.tight_layout()
plt.show()

display(pd.crosstab(metadata["State"], metadata["Species"]))

✏️ **Findings (write after running).** *Which training columns are not in the test set, and what does that mean for the model inputs? Which states, species or months have few images? Images are collected in batches on a few days: note this, it matters for validation in the next section. State the limitations clearly, then move on.*

## 8. Validation strategy

Images taken on the same day in the same place look alike. With a random split, such near-copies land in both the training and the validation part, so the validation score can be too optimistic. We therefore compare two strategies:

1. **Random split**: 80% training, 20% validation, seed 42.
2. **Grouped 5-fold cross-validation**: all images from one sampling date stay in the same fold. Every image is used for validation exactly once, so the score is also less dependent on one lucky split.

In [ ]:
train, validation = split_images(images, validation_ratio=0.20, seed=42)
print("Random split:", len(train), "training and", len(validation), "validation images")

shared_days = set(metadata.loc[train.index, "Sampling_Date"]) & set(metadata.loc[validation.index, "Sampling_Date"])
print("Sampling days that appear on both sides of the random split:", len(shared_days))

folds = group_folds(metadata["Sampling_Date"], n_splits=5)
display(pd.DataFrame({
    "Images": folds.value_counts().sort_index(),
    "Sampling days": metadata.groupby(folds)["Sampling_Date"].nunique(),
    "Mean Dry_Total_g": images.groupby(folds)["Dry_Total_g"].mean().round(1),
}).rename_axis("Fold"))

## 9. Baselines

1. **Training mean.** Predicts the training mean of each target for every image. Any useful model must beat it.
2. **RGB + Ridge.** Reduces each image to three numbers (mean red, green and blue), standardises them and fits a Ridge regression to the five targets. Negative predictions are clipped to zero.

Scaling and model parameters are learned from the training images only. We report the **Kaggle weighted R²** (computed over all targets together; weights 0.5 for total, 0.2 for GDM, 0.1 for the others) and the **mean absolute error** in grams per target. [Kaggle evaluation](https://www.kaggle.com/competitions/csiro-biomass/overview/evaluation)

In [ ]:
models = {
    "Training mean": mean_predictions,
    "RGB + Ridge": lambda train, index: rgb_ridge_predictions(rgb, train, index, alpha=1.0),
}

# Strategy 1: single random split.
random_split_r2 = pd.Series({name: kaggle_r2(validation, predict(train, validation.index))
                             for name, predict in models.items()})

# Strategy 2: grouped 5-fold cross-validation.
fold_scores, predictions = cross_validate(images, folds, models)
display(fold_scores.round(4))

results = pd.DataFrame({
    "Random split R²": random_split_r2,
    "Grouped CV R² (mean)": fold_scores.mean(),
    "Grouped CV R² (std)": fold_scores.std(),
    "Grouped CV R² (all images)": {name: kaggle_r2(images, predicted)
                                   for name, predicted in predictions.items()},
})
display(results.round(4))

In [ ]:
# Mean absolute error per target, from the out-of-fold predictions of the grouped CV.
mae = pd.DataFrame({name: (images - predicted).abs().mean()
                    for name, predicted in predictions.items()})
display(mae.round(2).rename_axis("Target · MAE (g)"))

# Predicted against actual total biomass for the Ridge baseline.
figure, axis = plt.subplots(figsize=(5, 5))
axis.scatter(images["Dry_Total_g"], predictions["RGB + Ridge"]["Dry_Total_g"], s=12, alpha=0.6)
limit = images["Dry_Total_g"].max()
axis.plot([0, limit], [0, limit], color="black", linewidth=1)
axis.set_xlabel("Actual Dry_Total_g")
axis.set_ylabel("Predicted Dry_Total_g (out of fold)")
axis.set_title("RGB + Ridge")
plt.tight_layout()
plt.show()

✏️ **Results (write after running).** *Report the weighted R² of both baselines under both strategies. Is the grouped score lower than the random-split score, and how much do the folds vary? For which targets does RGB + Ridge beat the mean, and where does it fail (look at clover and at high biomass in the scatter plot)?*

## 10. Conclusions and plan for Assignment 2

✏️ *Summarise in a few bullets, for example:*

- *Data quality: what was checked, what was found, what was kept.*
- *What the EDA implies for modelling: skewed targets, many zero clover values, strongly related targets, metadata not available at test time, few sampling days.*
- *Baseline scores to beat (grouped CV and Kaggle leaderboard).*
- *Ideas to try next: features from a pretrained image model, training on a log scale, predicting the three components and deriving GDM and total, image augmentation.*